# xNES learning-rate sweeps

This notebook tunes the xNES learning-rate knobs `eta_mean`, `eta_scale_global`, and `eta_scale_shape` with explicit center values defined at the top of the notebook.

- Each sweep changes one knob while the other two stay at their configured center values.
- Optimization is driven by noisy translated objectives.
- `sphere` provides the separable baseline.
- `ellipsoid` and `cigar` are rotated generically per case to stress shape adaptation without rotating every function.
- Each `function x dimension x trial` case reuses the same optimizer and noise RNG streams across eta settings.
- Scoring uses the clean objective at the current mean on log-spaced checkpoints through the run, averaged as a log-time AOC approximation.
- Each sweep also shows a null-ranking stability curve on a separate red axis.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "leitwerk").is_dir() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks.problems import translated_objective
from collections.abc import Callable
from dataclasses import replace
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

from leitwerk import XNES, XNESLearningRates, XNESStatus
from leitwerk.xnes import _default_sample_count

In [ ]:
MASTER_SEED = 1234

ETA_MEAN_CENTER = 1.0
ETA_SCALE_GLOBAL_CENTER = 0.5
ETA_SCALE_SHAPE_CENTER = 0.25

ETA_MEAN_LOG_MIN = -1
ETA_SCALE_GLOBAL_LOG_MIN = -1
ETA_SCALE_SHAPE_LOG_MIN = -1
RESOLUTION = 100
NUM_TRIALS = 3
NULL_TRIALS = 32
DIMENSION = [2, 3, 5, 8]
NOISE_SCALE = 1e-1

X0 = 0.0
SIGMA0 = 1.0

MAX_GENERATIONS = 128
NULL_GENERATIONS = 10
OFFSET_SCALE = 3.0
SCORE_EPS = 1e-12
SCORE_CLIP = 6.0

FUNCTION_NAMES = ("sphere", "ackley", "rosenbrock", "ellipsoid", "cigar")
ROTATED_FUNCTIONS = frozenset({"ellipsoid", "cigar"})
CENTER_LEARNING_RATES = XNESLearningRates(
    eta_mean=ETA_MEAN_CENTER,
    eta_scale_global=ETA_SCALE_GLOBAL_CENTER,
    eta_scale_shape=ETA_SCALE_SHAPE_CENTER,
)


def make_eta_sweep(log_min: int) -> np.ndarray:
    if RESOLUTION < 1:
        msg = "RESOLUTION must be at least 1 for a positive log sweep."
        raise ValueError(msg)
    sweep = np.logspace(log_min, 0.0, RESOLUTION, base=10.0, dtype=float)
    sweep[-1] = 1.0
    return sweep


def make_checkpoints(max_generations: int) -> list[int]:
    checkpoints: list[int] = []
    generation = 1
    while generation < max_generations:
        checkpoints.append(generation)
        generation *= 2
    checkpoints.append(max_generations)
    return checkpoints


ETA_SWEEPS = {
    "eta_mean": make_eta_sweep(ETA_MEAN_LOG_MIN),
    "eta_scale_global": make_eta_sweep(ETA_SCALE_GLOBAL_LOG_MIN),
    "eta_scale_shape": make_eta_sweep(ETA_SCALE_SHAPE_LOG_MIN),
}
CHECKPOINT_GENERATIONS = make_checkpoints(MAX_GENERATIONS)
CASES = list(product(FUNCTION_NAMES, DIMENSION, range(NUM_TRIALS)))
CASE_COUNT = len(CASES)

print(
    {
        "center_learning_rates": CENTER_LEARNING_RATES,
        "functions": FUNCTION_NAMES,
        "rotated_functions": sorted(ROTATED_FUNCTIONS),
        "dimensions": DIMENSION,
        "num_trials": NUM_TRIALS,
        "null_trials": NULL_TRIALS,
        "resolution": RESOLUTION,
        "max_generations": MAX_GENERATIONS,
        "null_generations": NULL_GENERATIONS,
        "checkpoint_generations": CHECKPOINT_GENERATIONS,
        "noise_scale": NOISE_SCALE,
        "cases_per_eta": CASE_COUNT,
    }
)

In [ ]:
def make_case_streams(function_name: str, dim: int, trial_index: int) -> tuple[np.random.SeedSequence, np.random.SeedSequence]:
    function_index = FUNCTION_NAMES.index(function_name)
    case_seed = np.random.SeedSequence([MASTER_SEED, function_index, dim, trial_index])
    problem_seed, optimizer_seed = case_seed.spawn(2)
    return problem_seed, optimizer_seed


def make_problem(
    function_name: str,
    dim: int,
    problem_seed: np.random.SeedSequence,
) -> tuple[Callable[[np.ndarray], np.ndarray], Callable[[np.ndarray], np.ndarray], np.ndarray, bool]:
    return translated_objective(function_name, dim, problem_seed, OFFSET_SCALE, NOISE_SCALE)


def log_improvement(initial_clean: float, current_clean: float) -> float:
    residual = (current_clean + SCORE_EPS) / (initial_clean + SCORE_EPS)
    residual = float(np.clip(residual, 10.0 ** -SCORE_CLIP, 10.0 ** SCORE_CLIP))
    return -float(np.log10(residual))


NULL_METRIC_LABELS = {
    "eta_mean": "null mean drift",
    "eta_scale_global": "null |log global scale|",
    "eta_scale_shape": "null shape deformation",
}


def shape_deformation(xnes: XNES) -> float:
    covariance = xnes.scale_shape @ xnes.scale_shape.T
    log_eigenvalues = np.log(np.linalg.eigvalsh(covariance))
    return float(np.sqrt(np.mean(log_eigenvalues**2)))


def run_null_trial(dim: int, trial_index: int, learning_rates: XNESLearningRates, knob: str) -> float:
    seed = np.random.SeedSequence([MASTER_SEED, 2_000_000, dim, trial_index])
    rng = np.random.default_rng(seed)
    xnes = XNES(np.full(dim, X0, dtype=float), SIGMA0)
    mean0 = xnes.mean.copy()
    log_scale_global0 = float(np.log(xnes.scale_global))
    batch_size = _default_sample_count(None, dim)
    max_drift = 0.0

    for _ in range(NULL_GENERATIONS):
        z = xnes.sample(batch_size, rng=rng)
        ranking = rng.permutation(batch_size).tolist()
        status = xnes.update(z, ranking, learning_rates)

        if knob == "eta_mean":
            drift = np.linalg.norm(xnes.mean - mean0) / (np.sqrt(dim) * SIGMA0)
        elif knob == "eta_scale_global":
            drift = abs(float(np.log(xnes.scale_global)) - log_scale_global0)
        else:
            drift = shape_deformation(xnes)
        max_drift = max(max_drift, float(drift))

        if status is not XNESStatus.OK:
            break

    return max_drift


def run_trial(
    function_name: str,
    dim: int,
    trial_index: int,
    learning_rates: XNESLearningRates,
) -> dict[str, float | int | str | bool]:
    problem_seed, optimizer_seed = make_case_streams(function_name, dim, trial_index)
    clean, noisy, offset, rotation_applied = make_problem(function_name, dim, problem_seed)
    xnes = XNES(np.full(dim, X0, dtype=float), SIGMA0)
    rng = np.random.default_rng(optimizer_seed)
    batch_size = _default_sample_count(None, dim)

    initial_clean = float(clean(xnes.mean)[0])
    current_clean = initial_clean
    status = XNESStatus.OK
    generations = 0
    checkpoint_scores: list[float] = []
    next_checkpoint_index = 0
    last_score = 0.0

    for generation in range(1, MAX_GENERATIONS + 1):
        z = xnes.sample(batch_size, rng=rng)
        x = xnes.transform(z)
        values = noisy(x)
        ranking = np.argsort(values, kind="stable").tolist()
        status = xnes.update(z, ranking, learning_rates)
        generations = generation

        current_clean = float(clean(xnes.mean)[0])
        last_score = log_improvement(initial_clean, current_clean)
        while next_checkpoint_index < len(CHECKPOINT_GENERATIONS) and generation >= CHECKPOINT_GENERATIONS[next_checkpoint_index]:
            checkpoint_scores.append(last_score)
            next_checkpoint_index += 1

        if status is not XNESStatus.OK:
            break

    while len(checkpoint_scores) < len(CHECKPOINT_GENERATIONS):
        checkpoint_scores.append(last_score)

    return {
        "function": function_name,
        "dimension": dim,
        "trial": trial_index,
        "rotated": rotation_applied,
        "offset_norm": float(np.linalg.norm(offset)),
        "batch_size": batch_size,
        "generations": generations,
        "status": status.name,
        "initial_clean": initial_clean,
        "final_clean": current_clean,
        "final_score": checkpoint_scores[-1],
        "score": float(np.mean(checkpoint_scores)),
    }

In [ ]:
results: list[dict[str, float | int | str | bool]] = []

for knob, sweep_values in ETA_SWEEPS.items():
    for eta_value in tqdm(sweep_values, desc=knob):
        learning_rates = replace(CENTER_LEARNING_RATES, **{knob: float(eta_value)})
        for function_name, dim, trial_index in CASES:
            row = run_trial(function_name, dim, trial_index, learning_rates)
            row.update({"knob": knob, "eta_value": float(eta_value)})
            results.append(row)

print(f"{len(results)=} ({CASE_COUNT} cases per eta)")

In [ ]:
null_results: list[dict[str, float | int | str]] = []

for knob, sweep_values in ETA_SWEEPS.items():
    for eta_value in tqdm(sweep_values, desc=f"null {knob}"):
        learning_rates = replace(CENTER_LEARNING_RATES, **{knob: float(eta_value)})
        for dim, trial_index in product(DIMENSION, range(NULL_TRIALS)):
            null_results.append(
                {
                    "knob": knob,
                    "eta_value": float(eta_value),
                    "dimension": dim,
                    "trial": trial_index,
                    "drift": run_null_trial(dim, trial_index, learning_rates, knob),
                }
            )

null_aggregate: dict[str, dict[str, np.ndarray]] = {}
null_case_count = len(DIMENSION) * NULL_TRIALS
for knob, sweep_values in ETA_SWEEPS.items():
    drifts = np.array(
        [float(row["drift"]) for row in null_results if row["knob"] == knob],
        dtype=float,
    ).reshape(len(sweep_values), null_case_count)
    null_aggregate[knob] = {
        "median": np.median(drifts, axis=1),
        "q95": np.quantile(drifts, 0.95, axis=1),
    }

print(f"{len(null_results)=} ({null_case_count} null cases per eta)")

In [ ]:
aggregate: dict[str, dict[str, np.ndarray]] = {}
best_by_knob: list[dict[str, float | str | tuple[float, float]]] = []

for knob, sweep_values in ETA_SWEEPS.items():
    knob_rows = [row for row in results if row["knob"] == knob]
    scores = np.array([float(row["score"]) for row in knob_rows], dtype=float).reshape(len(sweep_values), CASE_COUNT)
    failures = np.array([XNESStatus[str(row["status"])].is_error for row in knob_rows], dtype=float).reshape(len(sweep_values), CASE_COUNT)

    stats = {
        "eta_values": sweep_values,
        "mean": scores.mean(axis=1),
        "q05": np.quantile(scores, 0.05, axis=1),
        "q25": np.quantile(scores, 0.25, axis=1),
        "q75": np.quantile(scores, 0.75, axis=1),
        "q95": np.quantile(scores, 0.95, axis=1),
        "failure_rate": failures.mean(axis=1),
    }
    aggregate[knob] = stats

    best_index = int(np.argmax(stats["mean"]))
    best_by_knob.append(
        {
            "knob": knob,
            "best_eta": float(stats["eta_values"][best_index]),
            "mean_aoc_score": float(stats["mean"][best_index]),
            "iqr": (
                float(stats["q25"][best_index]),
                float(stats["q75"][best_index]),
            ),
            "tail_band": (
                float(stats["q05"][best_index]),
                float(stats["q95"][best_index]),
            ),
            "failure_rate": float(stats["failure_rate"][best_index]),
        }
    )

best_by_knob

## Sweep plots

Each plot aggregates the checkpoint-averaged log-time AOC score across all dimensions, functions, and trials for one knob.

The red secondary axis shows the maximum excursion under uniformly random rankings over `NULL_GENERATIONS` updates, aggregated across dimensions and null trials. Lower is more stable.

- Dark band: interquartile range (25%-75%)
- Light band: tail band (5%-95%)
- Dashed line: configured center value for that knob
- Red solid line: median null excursion
- Red dotted line: 95th-percentile null excursion

In [ ]:
for knob in ("eta_mean", "eta_scale_global", "eta_scale_shape"):
    stats = aggregate[knob]
    eta_values = stats["eta_values"]
    center_eta = getattr(CENTER_LEARNING_RATES, knob)

    null_stats = null_aggregate[knob]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.fill_between(eta_values, stats["q05"], stats["q95"], color="#4c72b0", alpha=0.16, label="5%-95%")
    ax.fill_between(eta_values, stats["q25"], stats["q75"], color="#4c72b0", alpha=0.32, label="25%-75%")
    ax.plot(eta_values, stats["mean"], color="#1f2a44", linewidth=2.2, label="mean AOC")
    ax.axvline(center_eta, color="#c44e52", linestyle="--", linewidth=1.5, label=f"center = {center_eta:g}")

    # ax.set_xscale("log")
    ax.set_xlim(float(np.min(eta_values)), float(np.max(eta_values)))
    ax.set_xlabel("learning rate")
    ax.set_ylabel("AOC score")
    ax.set_title(f"xNES {knob} sweep")
    ax.grid(True, which="both", alpha=0.3)

    null_ax = ax.twinx()
    null_ax.plot(
        eta_values,
        null_stats["median"],
        color="#c44e52",
        linewidth=1.5,
        alpha=0.75,
        label="null median",
    )
    null_ax.plot(
        eta_values,
        null_stats["q95"],
        color="#c44e52",
        linestyle=":",
        linewidth=1.5,
        alpha=0.5,
        label="null 95%",
    )
    null_ax.set_ylabel(NULL_METRIC_LABELS[knob], color="#c44e52")
    null_ax.tick_params(axis="y", colors="#c44e52")

    score_handles, score_labels = ax.get_legend_handles_labels()
    null_handles, null_labels = null_ax.get_legend_handles_labels()
    ax.legend(score_handles + null_handles, score_labels + null_labels, loc="best")
    plt.show()

These sweeps are exploratory training experiments, not held-out comparisons. Bands show variation across cases, not confidence intervals. Failure rates count numerical/error statuses; convergence-like early stops are separate from failures. Shared landscape/noise helpers preserve this notebook's original half-normal experiment. Learning-rate choices have not been retuned.